# LAHAJA External Paired Telephony Replication

This one-click GPU Colab notebook tests whether the CallWhisper-8k Vaani finding generalizes to an external Hindi accent benchmark.

It selects one deterministic 1-30 second utterance from each of LAHAJA's 132 speakers, builds five matched audio conditions, and evaluates two pinned Hindi Whisper checkpoints through identical decoding:

- `ARTPARK-IISc/whisper-medium-vaani-hindi`
- `adalat-ai/whisper-small-hi-high-lr`

Primary question:

> Does telephone degradation impose a larger WER penalty on the compact Adalat Whisper-small model than on ARTPARK Whisper-medium?

The notebook uses conventional single-reference WER/CER, immediate Drive checkpoints, and 20,000 speaker-clustered bootstrap replicates. It does not train either model. Absolute WER is not directly comparable with the earlier Vaani multi-reference score; paired channel deltas are the replication target.


## Before Running

1. Open [ai4bharat/Lahaja](https://huggingface.co/datasets/ai4bharat/Lahaja), accept the gated access conditions, and wait for access.
2. Create a Hugging Face read token.
3. In Colab, open **Secrets**, add `HF_TOKEN`, and enable notebook access.
4. Select **Runtime > Change runtime type > T4 GPU**.
5. Run all cells.

Persistent outputs go to:

```text
MyDrive/call-whisper/results/lahaja_paired_external_v1/
```

Source and transformed audio remain in your private Drive artifacts. Do not commit or redistribute LAHAJA audio.


In [ ]:
# Environment setup: mount Drive, clone from a valid working directory, and install pinned ranges.
import importlib
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
if subprocess.run(['nvidia-smi'], check=False).returncode:
    raise RuntimeError('GPU runtime required. Select Runtime > Change runtime type > T4 GPU.')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
REPO_DIR = Path('/content/CallWhisper-8k')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'datasets>=4.0,<5', 'huggingface_hub>=0.34,<2',
    'transformers>=4.46,<5', 'accelerate>=1,<2',
    'jiwer>=3,<5', 'numpy>=1.26,<3', 'pandas>=2,<3',
    'soundfile>=0.12', 'tabulate>=0.9', 'tqdm>=4.66',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'
], check=True)
importlib.invalidate_caches()

_paired = importlib.import_module('callwhisper.datasets.paired_telephony')
_bootstrap = importlib.import_module('callwhisper.eval.paired_bootstrap')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Python:', platform.python_version())
print('Repository commit:', commit)
print('Paired audio module:', _paired.__file__)
print('Bootstrap module:', _bootstrap.__file__)


In [ ]:
# Frozen external-replication configuration.
import hashlib
import json
from importlib.metadata import version

from google.colab import userdata
from huggingface_hub import HfApi, get_token, login

DATASET_ID = 'ai4bharat/Lahaja'
DATASET_REVISION = 'd4ffd2ecbdd933e37c917ddcf620eef159ceb3a7'
SPLIT = 'test'
EXPECTED_DATASET_ROWS = 6152
EXPECTED_UNIQUE_SPEAKERS = 132
SPEAKER_LIMIT = 132
MIN_DURATION_S = 1.0
MAX_DURATION_S = 30.0
SEED = 0
NUM_BEAMS = 1
LANGUAGE = 'hi'
TASK = 'transcribe'
BOOTSTRAP_REPLICATES = 20_000

CONDITIONS = (
    'original',
    'bandlimit_8k',
    'bandlimit_8k_g711_alaw',
    'bandlimit_8k_g711_mulaw',
    'bandlimit_8k_gsm_fr',
)
MODELS = (
    {
        'label': 'artpark_medium_vaani_hindi',
        'model_id': 'ARTPARK-IISc/whisper-medium-vaani-hindi',
        'revision': '8e4d906e0eec66f27a31286e1a034702ef6d11bc',
    },
    {
        'label': 'adalat_whisper_small_hi_high_lr',
        'model_id': 'adalat-ai/whisper-small-hi-high-lr',
        'revision': 'e78553113fe7a483dbf82fefb2cbe4ea4b6bf901',
    },
)

WORK_ROOT = Path('/content/lahaja_paired_external_v1')
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'results' / 'lahaja_paired_external_v1'
SOURCE_DIR = WORK_ROOT / 'source_audio'
PAIRED_AUDIO_DIR = WORK_ROOT / 'paired_audio'
ARCHIVE_DIR = OUTPUT_DIR / 'archives'
for directory in (WORK_ROOT, OUTPUT_DIR, SOURCE_DIR, PAIRED_AUDIO_DIR, ARCHIVE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

try:
    secret = userdata.get('HF_TOKEN')
except Exception:
    secret = None
if secret:
    login(token=secret, add_to_git_credential=False)
token = get_token()
if not token:
    raise RuntimeError('HF_TOKEN missing. Add read token to Colab Secrets and enable access.')

api = HfApi(token=token)
try:
    dataset_info = api.dataset_info(DATASET_ID, revision=DATASET_REVISION)
except Exception as exc:
    raise RuntimeError(
        'LAHAJA access failed. Accept dataset conditions, then verify HF_TOKEN.'
    ) from exc
if dataset_info.sha != DATASET_REVISION:
    raise RuntimeError(f'Dataset revision mismatch: {dataset_info.sha}')

semantic_config = {
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'split': SPLIT,
    'expected_dataset_rows': EXPECTED_DATASET_ROWS,
    'expected_unique_speakers': EXPECTED_UNIQUE_SPEAKERS,
    'speaker_limit': SPEAKER_LIMIT,
    'duration_range_s': [MIN_DURATION_S, MAX_DURATION_S],
    'selection': 'one deterministic eligible row per speaker; scenario and state are reported',
    'reference': 'normalized, falling back to text',
    'conditions': list(CONDITIONS),
    'channel_semantics': 'bandlimit first, then codec for every codec condition',
    'models': list(MODELS),
    'num_beams': NUM_BEAMS,
    'language': LANGUAGE,
    'task': TASK,
    'seed': SEED,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'training_allowed': False,
    'training_disjointness': 'not proven; interpret as external benchmark replication',
}
fingerprint_paths = [
    REPO_DIR / 'src/callwhisper/datasets/paired_telephony.py',
    REPO_DIR / 'src/callwhisper/eval/multiref.py',
    REPO_DIR / 'src/callwhisper/eval/paired_bootstrap.py',
    REPO_DIR / 'notebooks/13_lahaja_external_paired_replication_colab.ipynb',
]
code_digest = hashlib.sha256()
for fingerprint_path in fingerprint_paths:
    code_digest.update(fingerprint_path.relative_to(REPO_DIR).as_posix().encode('utf-8'))
    code_digest.update(fingerprint_path.read_bytes())
code_fingerprint = code_digest.hexdigest()

config_path = OUTPUT_DIR / 'run_config.json'
if config_path.exists():
    existing_config = json.loads(config_path.read_text(encoding='utf-8'))
    if existing_config['semantic_config'] != semantic_config:
        raise RuntimeError(f'Existing output has a different frozen config: {config_path}')
    if existing_config['code_fingerprint'] != code_fingerprint:
        raise RuntimeError(
            'Relevant notebook/scoring code changed since this run started. '
            'Use a new output directory or restore the original code before resuming.'
        )
else:
    config_path.write_text(
        json.dumps(
            {
                'semantic_config': semantic_config,
                'repo_commit': commit,
                'code_fingerprint': code_fingerprint,
            },
            ensure_ascii=False, indent=2,
        ) + '\n',
        encoding='utf-8',
    )

package_versions = {
    name: version(name)
    for name in (
        'datasets', 'huggingface_hub', 'transformers', 'accelerate',
        'jiwer', 'numpy', 'pandas', 'soundfile', 'tqdm',
    )
}
package_versions.update({'python': platform.python_version(), 'repo_commit': commit})
(OUTPUT_DIR / 'package_versions.json').write_text(
    json.dumps(package_versions, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(semantic_config, indent=2))
print('Persistent output:', OUTPUT_DIR)


In [ ]:
# Download pinned LAHAJA and disable audio decoding to avoid TorchCodec dependency failures.
from datasets import Audio, load_dataset

print('Downloading pinned LAHAJA revision. First run downloads about 1.42 GB.')
dataset = load_dataset(
    DATASET_ID,
    split=SPLIT,
    revision=DATASET_REVISION,
    token=token,
    cache_dir='/content/huggingface_cache',
)
required_columns = {
    'audio_filepath', 'text', 'normalized', 'duration', 'scenario', 'fname',
    'native_language', 'gender', 'age_group', 'native_state',
    'native_district', 'sp_id',
}
missing_columns = required_columns - set(dataset.column_names)
if missing_columns:
    raise RuntimeError(f'LAHAJA schema changed; missing columns: {sorted(missing_columns)}')
if len(dataset) != EXPECTED_DATASET_ROWS:
    raise RuntimeError(
        f'Pinned dataset row count changed: {len(dataset)} != {EXPECTED_DATASET_ROWS}'
    )
if not isinstance(dataset.features['audio_filepath'], Audio):
    raise RuntimeError('audio_filepath is no longer an Audio feature')
dataset = dataset.cast_column('audio_filepath', Audio(decode=False))
print('Rows:', len(dataset))
print('Pinned revision:', dataset_info.sha)
print('Columns:', dataset.column_names)
print('Features:', dataset.features)


In [ ]:
# Build metadata inventory and freeze one eligible utterance per speaker.
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from callwhisper.datasets.paired_telephony import (
    deterministic_stratified_sample,
    normalize_group,
    sample_key,
    stable_key,
)

metadata_dataset = dataset.remove_columns('audio_filepath')
inventory_rows = []
for index, row in enumerate(tqdm(
    metadata_dataset, total=len(metadata_dataset), desc='Inventorying LAHAJA'
)):
    reference = str(row.get('normalized') or row.get('text') or '').strip()
    source_id = str(row.get('fname') or f'row-{index:06d}')
    duration = float(row.get('duration') or 0.0)
    inventory_rows.append({
        'dataset_index': index,
        'source_id': source_id,
        'sample_key': sample_key(DATASET_ID, source_id),
        'speaker_id': normalize_group(row.get('sp_id')),
        'reference_text': reference,
        'text': str(row.get('text') or '').strip(),
        'normalized': str(row.get('normalized') or '').strip(),
        'duration_s_metadata': duration,
        'scenario': normalize_group(row.get('scenario')),
        'native_language': normalize_group(row.get('native_language')),
        'gender': normalize_group(row.get('gender')),
        'age_group': normalize_group(row.get('age_group')),
        'state': normalize_group(row.get('native_state')),
        'district': normalize_group(row.get('native_district')),
    })

inventory_df = pd.DataFrame(inventory_rows)
eligible_df = inventory_df[
    inventory_df['reference_text'].ne('')
    & inventory_df['speaker_id'].ne('unknown')
    & inventory_df['duration_s_metadata'].between(
        MIN_DURATION_S, MAX_DURATION_S, inclusive='both'
    )
].copy()
eligible_speakers = eligible_df['speaker_id'].nunique()
if inventory_df['speaker_id'].replace('unknown', pd.NA).dropna().nunique() != EXPECTED_UNIQUE_SPEAKERS:
    raise RuntimeError('Pinned LAHAJA unique-speaker count no longer equals 132')
if eligible_speakers < SPEAKER_LIMIT:
    raise RuntimeError(
        f'Only {eligible_speakers} speakers have eligible 1-30 s clips; expected {SPEAKER_LIMIT}'
    )
if inventory_df['sample_key'].duplicated().any():
    raise RuntimeError('Duplicate source IDs/sample keys detected')

selected_rows = deterministic_stratified_sample(
    eligible_df.to_dict('records'),
    SPEAKER_LIMIT,
    SEED,
    speaker_column='speaker_id',
    stratum_columns=('scenario', 'state'),
)
pilot_df = pd.DataFrame(selected_rows).sort_values('sample_key').reset_index(drop=True)
if len(pilot_df) != SPEAKER_LIMIT or pilot_df['speaker_id'].nunique() != SPEAKER_LIMIT:
    raise RuntimeError('Frozen pilot is not one row per requested speaker')

inventory_df.to_csv(OUTPUT_DIR / 'lahaja_full_inventory.csv', index=False)
pilot_df.to_csv(OUTPUT_DIR / 'lahaja_speaker_132.csv', index=False)
selection_summary = {
    'inventory_rows': int(len(inventory_df)),
    'eligible_rows': int(len(eligible_df)),
    'eligible_speakers': int(eligible_speakers),
    'selected_rows': int(len(pilot_df)),
    'unique_speakers': int(pilot_df['speaker_id'].nunique()),
    'sample_key_set_sha256': stable_key(*sorted(pilot_df['sample_key'])),
    'scenario_counts': pilot_df['scenario'].value_counts().sort_index().to_dict(),
    'gender_counts': pilot_df['gender'].value_counts().sort_index().to_dict(),
    'state_counts': pilot_df['state'].value_counts().sort_index().to_dict(),
}
(OUTPUT_DIR / 'selection_summary.json').write_text(
    json.dumps(selection_summary, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
display(pilot_df.groupby(['scenario', 'gender']).size().rename('files').reset_index())
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


In [ ]:
# Restart-safe archive helpers and source-audio export without TorchCodec decoding.
import tarfile

from callwhisper.datasets.paired_telephony import sha256_file


def safe_extract(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        members = archive.getmembers()
        for member in members:
            if member.issym() or member.islnk():
                raise RuntimeError(f'Refusing archive link: {member.name}')
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe archive path: {member.name}')
        for member in tqdm(members, desc=f'Restoring {archive_path.name}'):
            archive.extract(member, destination, filter='data')


def copy_with_progress(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    total = source.stat().st_size
    with source.open('rb') as input_handle, temporary.open('wb') as output_handle, tqdm(
        total=total, unit='B', unit_scale=True, desc=f'Copying {destination.name}'
    ) as progress:
        while chunk := input_handle.read(8 * 1024 * 1024):
            output_handle.write(chunk)
            progress.update(len(chunk))
    temporary.replace(destination)


def archive_directory(source_dir: Path, archive_path: Path, arcname: str) -> None:
    temporary = Path('/content') / f'{archive_path.name}.part'
    if temporary.exists():
        temporary.unlink()
    files = sorted(path for path in source_dir.rglob('*') if path.is_file())
    with tarfile.open(temporary, 'w:gz') as archive:
        for path in tqdm(files, desc=f'Archiving {arcname}'):
            archive.add(path, arcname=str(Path(arcname) / path.relative_to(source_dir)))
    copy_with_progress(temporary, archive_path)
    temporary.unlink()
    print('Saved archive:', archive_path, 'GB=', round(archive_path.stat().st_size / 1e9, 3))


def restore_if_available(archive_path: Path) -> bool:
    if archive_path.exists():
        print('Restoring completed stage:', archive_path)
        safe_extract(archive_path, WORK_ROOT)
        return True
    return False


source_archive = ARCHIVE_DIR / 'source_audio.tar.gz'
restore_if_available(source_archive)
source_archive_needs_refresh = (
    len(list(SOURCE_DIR.glob('*'))) != SPEAKER_LIMIT
)
source_records = []
for row in tqdm(pilot_df.to_dict('records'), desc='Exporting selected source audio'):
    payload = dataset[int(row['dataset_index'])]['audio_filepath']
    original_name = str(payload.get('path') or '') if isinstance(payload, dict) else ''
    suffix = Path(original_name).suffix.lower() or '.audio'
    destination = SOURCE_DIR / f"{row['sample_key']}{suffix}"
    if not destination.exists():
        if isinstance(payload, dict) and payload.get('bytes') is not None:
            destination.write_bytes(bytes(payload['bytes']))
        elif isinstance(payload, dict) and payload.get('path') and Path(payload['path']).exists():
            shutil.copy2(payload['path'], destination)
        else:
            raise RuntimeError(f'Audio payload has neither bytes nor local path: {payload}')
    source_records.append({
        'sample_key': row['sample_key'],
        'source_audio_path': str(destination.relative_to(WORK_ROOT)),
        'source_sha256': sha256_file(destination),
    })

source_df = pd.DataFrame(source_records)
if len(source_df) != SPEAKER_LIMIT:
    raise RuntimeError('Source export row count mismatch')
source_df.to_csv(OUTPUT_DIR / 'source_audio_manifest.csv', index=False)
if source_archive_needs_refresh:
    if source_archive.exists():
        source_archive.unlink()
    archive_directory(SOURCE_DIR, source_archive, 'source_audio')
print('Source clips ready:', len(list(SOURCE_DIR.glob('*'))))


In [ ]:
# Generate five matched channel conditions and checkpoint each completed condition to Drive.
from callwhisper.datasets.paired_telephony import (
    ffmpeg_version,
    probe_audio,
    transform_audio,
    validate_codec_support,
)

print(ffmpeg_version())
print('Codec support:', validate_codec_support(CONDITIONS))
source_lookup = source_df.set_index('sample_key').to_dict('index')
all_transform_rows = []

for condition in CONDITIONS:
    condition_dir = PAIRED_AUDIO_DIR / condition
    condition_dir.mkdir(parents=True, exist_ok=True)
    condition_archive = ARCHIVE_DIR / f'{condition}.tar.gz'
    restore_if_available(condition_archive)
    condition_archive_needs_refresh = (
        len(list(condition_dir.glob('*.wav'))) != SPEAKER_LIMIT
    )
    condition_rows = []

    for row in tqdm(pilot_df.to_dict('records'), desc=f'Building {condition}'):
        source_path = WORK_ROOT / source_lookup[row['sample_key']]['source_audio_path']
        output_path = condition_dir / f"{row['sample_key']}.wav"
        if output_path.exists():
            output_probe = probe_audio(output_path)
            source_probe = probe_audio(source_path)
            metadata = {
                'condition': condition,
                'source_sha256': sha256_file(source_path),
                'output_sha256': sha256_file(output_path),
                'source_duration_s': source_probe['duration_s'],
                'duration_s': output_probe['duration_s'],
                'duration_delta_s': abs(
                    output_probe['duration_s'] - source_probe['duration_s']
                ),
                'sample_rate_hz': output_probe['sample_rate_hz'],
                'channels': output_probe['channels'],
            }
        else:
            metadata = transform_audio(source_path, output_path, condition)

        condition_rows.append({
            **row,
            **metadata,
            'audio_path': str(output_path.relative_to(WORK_ROOT)),
            'dataset_revision': DATASET_REVISION,
        })

    condition_df = pd.DataFrame(condition_rows)
    condition_df.to_csv(
        OUTPUT_DIR / f'lahaja_speaker_132_{condition}.csv', index=False
    )
    all_transform_rows.extend(condition_rows)
    if condition_archive_needs_refresh:
        if condition_archive.exists():
            condition_archive.unlink()
        archive_directory(
            condition_dir, condition_archive, f'paired_audio/{condition}'
        )
    print('Completed:', condition, len(condition_df), 'files')

paired_df = pd.DataFrame(all_transform_rows)
paired_df.to_csv(OUTPUT_DIR / 'lahaja_speaker_132_all_conditions.csv', index=False)

counts = paired_df.groupby('sample_key')['condition'].nunique()
duration_tolerance = paired_df['source_duration_s'].mul(0.01).clip(lower=0.05)
validation = {
    'source_rows': int(len(pilot_df)),
    'paired_rows': int(len(paired_df)),
    'expected_paired_rows': int(SPEAKER_LIMIT * len(CONDITIONS)),
    'unique_speakers': int(pilot_df['speaker_id'].nunique()),
    'samples_with_all_conditions': int((counts == len(CONDITIONS)).sum()),
    'sample_rates_hz': sorted(int(value) for value in paired_df['sample_rate_hz'].unique()),
    'channels': sorted(int(value) for value in paired_df['channels'].unique()),
    'max_duration_delta_s': float(paired_df['duration_delta_s'].max()),
    'duration_tolerance_violations': int(
        (paired_df['duration_delta_s'] > duration_tolerance).sum()
    ),
    'missing_output_files': int(sum(
        not (WORK_ROOT / path).exists() for path in paired_df['audio_path']
    )),
    'dataset_revision': DATASET_REVISION,
    'repo_commit': commit,
    'code_fingerprint': code_fingerprint,
    'channel_semantics': 'bandlimit first, then codec for every codec condition',
}
assert validation['paired_rows'] == validation['expected_paired_rows']
assert validation['unique_speakers'] == SPEAKER_LIMIT
assert validation['samples_with_all_conditions'] == SPEAKER_LIMIT
assert validation['sample_rates_hz'] == [16000]
assert validation['channels'] == [1]
assert validation['duration_tolerance_violations'] == 0
assert validation['missing_output_files'] == 0
(OUTPUT_DIR / 'validation_summary.json').write_text(
    json.dumps(validation, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(validation, indent=2))


In [ ]:
# Listening sanity check. Confirm all five rows contain the same utterance.
from IPython.display import Audio as NotebookAudio
from IPython.display import Markdown, display

listening_key = pilot_df.iloc[0]['sample_key']
listening_rows = paired_df[paired_df['sample_key'] == listening_key].copy()
listening_rows['condition'] = pd.Categorical(
    listening_rows['condition'], categories=CONDITIONS, ordered=True
)
listening_rows = listening_rows.sort_values('condition')
reference_text = listening_rows.iloc[0]['reference_text']
display(Markdown(f'**Reference:** {reference_text}'))
for row in listening_rows.to_dict('records'):
    display(Markdown(f"### {row['condition']}"))
    display(NotebookAudio(str(WORK_ROOT / row['audio_path'])))
print('Expected: same words/speaker in every condition; GSM should sound most degraded.')


In [ ]:
# Shared inference and single-reference scoring path.
import gc
import time

import soundfile as sf
import torch
from jiwer import cer as jiwer_cer
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

from callwhisper.eval.multiref import normalize_vaani_text, single_reference_score

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('CUDA GPU required')
DTYPE = torch.float16
print('GPU:', torch.cuda.get_device_name(0))

eval_df = paired_df.copy()
eval_df['condition'] = pd.Categorical(
    eval_df['condition'], categories=CONDITIONS, ordered=True
)
eval_df = eval_df.sort_values(['sample_key', 'condition']).reset_index(drop=True)
eval_df['resolved_audio_path'] = eval_df['audio_path'].map(
    lambda path: str(WORK_ROOT / path)
)
assert len(eval_df) == SPEAKER_LIMIT * len(CONDITIONS)
assert eval_df.groupby('sample_key')['condition'].nunique().eq(len(CONDITIONS)).all()
assert all(Path(path).exists() for path in eval_df['resolved_audio_path'])
eval_df.to_csv(OUTPUT_DIR / 'frozen_eval_rows.csv', index=False)


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line_number, line in enumerate(
        path.read_text(encoding='utf-8').splitlines(), start=1
    ):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f'Invalid JSONL at {path}:{line_number}') from exc
    return rows


def append_jsonl(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        handle.flush()


def transcribe_one(model, processor, audio_path: Path) -> tuple[str, float]:
    audio, sample_rate = sf.read(audio_path, dtype='float32', always_2d=False)
    if sample_rate != 16000:
        raise ValueError(f'Expected 16 kHz, got {sample_rate}: {audio_path}')
    if getattr(audio, 'ndim', 1) != 1:
        raise ValueError(f'Expected mono audio: {audio_path}')
    inputs = processor(audio, sampling_rate=sample_rate, return_tensors='pt')
    input_features = inputs.input_features.to(device=DEVICE, dtype=DTYPE)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features=input_features,
            language=LANGUAGE,
            task=TASK,
            num_beams=NUM_BEAMS,
            do_sample=False,
            max_new_tokens=225,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    hypothesis = processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0].strip()
    return hypothesis, elapsed


def evaluate_model(spec: dict) -> Path:
    output_path = OUTPUT_DIR / f"{spec['label']}_predictions.jsonl"
    existing = read_jsonl(output_path)
    complete = {(row['sample_key'], row['condition']) for row in existing}
    expected = {
        (row.sample_key, str(row.condition))
        for row in eval_df.itertuples()
    }
    if complete == expected:
        print(spec['label'], 'already complete; skipping model load.')
        return output_path
    if not complete.issubset(expected):
        raise RuntimeError(f'Unexpected rows in checkpoint: {output_path}')

    print('Loading', spec['model_id'], 'revision', spec['revision'])
    processor = AutoProcessor.from_pretrained(
        spec['model_id'], revision=spec['revision']
    )
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        spec['model_id'],
        revision=spec['revision'],
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    ).to(DEVICE)
    model.eval()
    model.generation_config.forced_decoder_ids = None

    pending = [
        row for row in eval_df.to_dict('records')
        if (row['sample_key'], str(row['condition'])) not in complete
    ]
    print(spec['label'], 'pending rows:', len(pending), '/', len(eval_df))
    for row in tqdm(pending, desc=spec['label']):
        hypothesis, elapsed = transcribe_one(
            model, processor, Path(row['resolved_audio_path'])
        )
        reference = row['reference_text']
        score = single_reference_score(reference, hypothesis)
        normalized_reference = normalize_vaani_text(reference)
        normalized_hypothesis = normalize_vaani_text(hypothesis)
        duration = float(row['duration_s'])
        append_jsonl(output_path, {
            'model_label': spec['label'],
            'model_id': spec['model_id'],
            'model_revision': spec['revision'],
            'sample_key': row['sample_key'],
            'speaker_id': str(row['speaker_id']),
            'scenario': row['scenario'],
            'native_language': row['native_language'],
            'gender': row['gender'],
            'age_group': row['age_group'],
            'state': row['state'],
            'district': row['district'],
            'condition': str(row['condition']),
            'audio_path': row['audio_path'],
            'reference_text': reference,
            'hypothesis_text': hypothesis,
            'substitutions': score.substitutions,
            'insertions': score.insertions,
            'deletions': score.deletions,
            'errors': score.errors,
            'reference_words': score.reference_words,
            'wer': score.wer,
            'cer': float(jiwer_cer(normalized_reference, normalized_hypothesis)),
            'duration_s': duration,
            'inference_s': elapsed,
            'real_time_factor': elapsed / duration if duration > 0 else None,
            'num_beams': NUM_BEAMS,
            'language': LANGUAGE,
            'repo_commit': commit,
            'code_fingerprint': code_fingerprint,
        })

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    completed_rows = len(read_jsonl(output_path))
    print('Completed:', output_path, 'rows=', completed_rows)
    if completed_rows != len(eval_df):
        raise RuntimeError(f'Prediction checkpoint incomplete: {completed_rows}/{len(eval_df)}')
    return output_path


In [ ]:
# Run both models. Every completed prediction is already safe in Drive.
prediction_paths = [evaluate_model(spec) for spec in MODELS]
print('Prediction checkpoints:')
for path in prediction_paths:
    print('-', path)


In [ ]:
# Build per-condition metrics, model comparison, and paired utterance deltas.
prediction_rows = [
    row for path in prediction_paths for row in read_jsonl(path)
]
predictions_df = pd.DataFrame(prediction_rows)
expected_total = len(MODELS) * SPEAKER_LIMIT * len(CONDITIONS)
assert len(predictions_df) == expected_total
assert not predictions_df.duplicated(
    ['model_label', 'sample_key', 'condition']
).any()


def summarize_group(frame: pd.DataFrame, label: str) -> dict:
    substitutions = int(frame['substitutions'].sum())
    insertions = int(frame['insertions'].sum())
    deletions = int(frame['deletions'].sum())
    reference_words = int(frame['reference_words'].sum())
    references = [
        normalize_vaani_text(text) for text in frame['reference_text'].tolist()
    ]
    hypotheses = [
        normalize_vaani_text(text) for text in frame['hypothesis_text'].tolist()
    ]
    return {
        'model_label': frame['model_label'].iloc[0],
        'slice': label,
        'speakers': int(frame['sample_key'].nunique()),
        'files': int(len(frame)),
        'substitutions': substitutions,
        'insertions': insertions,
        'deletions': deletions,
        'reference_words': reference_words,
        'corpus_wer': (substitutions + insertions + deletions) / reference_words,
        'macro_utterance_wer': float(frame['wer'].mean()),
        'corpus_cer': float(jiwer_cer(references, hypotheses)),
        'macro_utterance_cer': float(frame['cer'].mean()),
        'mean_real_time_factor': float(frame['real_time_factor'].mean()),
    }


summary_rows = []
for model_label, model_frame in predictions_df.groupby('model_label', sort=False):
    for condition in CONDITIONS:
        condition_frame = model_frame[model_frame['condition'] == condition]
        summary_rows.append(summarize_group(condition_frame, condition))
    summary_rows.append(summarize_group(
        model_frame[model_frame['condition'] != 'original'],
        'pooled_telephone',
    ))
summary_df = pd.DataFrame(summary_rows)

model_labels = [spec['label'] for spec in MODELS]
comparison_df = summary_df.pivot(
    index='slice',
    columns='model_label',
    values=['corpus_wer', 'corpus_cer', 'mean_real_time_factor'],
)
comparison_df.columns = [
    f'{metric}_{model}' for metric, model in comparison_df.columns
]
comparison_df = comparison_df.reset_index()
comparison_df['adalat_minus_artpark_wer'] = (
    comparison_df[f'corpus_wer_{model_labels[1]}']
    - comparison_df[f'corpus_wer_{model_labels[0]}']
)
comparison_df['adalat_minus_artpark_cer'] = (
    comparison_df[f'corpus_cer_{model_labels[1]}']
    - comparison_df[f'corpus_cer_{model_labels[0]}']
)

paired_detail_rows = []
paired_summary_rows = []
for model_label, model_frame in predictions_df.groupby('model_label', sort=False):
    original = model_frame[model_frame['condition'] == 'original'][
        ['sample_key', 'wer', 'cer', 'errors']
    ].rename(columns={
        'wer': 'original_wer',
        'cer': 'original_cer',
        'errors': 'original_errors',
    })
    for condition in CONDITIONS:
        if condition == 'original':
            continue
        degraded = model_frame[model_frame['condition'] == condition][
            [
                'sample_key', 'speaker_id', 'scenario', 'gender', 'state',
                'wer', 'cer', 'errors',
            ]
        ].rename(columns={
            'wer': 'condition_wer',
            'cer': 'condition_cer',
            'errors': 'condition_errors',
        })
        paired = degraded.merge(original, on='sample_key', validate='one_to_one')
        paired['model_label'] = model_label
        paired['condition'] = condition
        paired['wer_delta_vs_original'] = (
            paired['condition_wer'] - paired['original_wer']
        )
        paired['cer_delta_vs_original'] = (
            paired['condition_cer'] - paired['original_cer']
        )
        paired['error_delta_vs_original'] = (
            paired['condition_errors'] - paired['original_errors']
        )
        paired_detail_rows.extend(paired.to_dict('records'))
        delta = paired['wer_delta_vs_original']
        paired_summary_rows.append({
            'model_label': model_label,
            'condition': condition,
            'speakers': int(len(paired)),
            'improved': int((delta < -1e-12).sum()),
            'unchanged': int((delta.abs() <= 1e-12).sum()),
            'worsened': int((delta > 1e-12).sum()),
            'mean_utterance_wer_delta': float(delta.mean()),
            'median_utterance_wer_delta': float(delta.median()),
            'mean_utterance_cer_delta': float(
                paired['cer_delta_vs_original'].mean()
            ),
            'total_error_delta': int(paired['error_delta_vs_original'].sum()),
        })

paired_detail_df = pd.DataFrame(paired_detail_rows).sort_values(
    ['model_label', 'condition', 'wer_delta_vs_original', 'sample_key']
)
paired_summary_df = pd.DataFrame(paired_summary_rows)

summary_df.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
(OUTPUT_DIR / 'summary.md').write_text(
    summary_df.to_markdown(index=False) + '\n', encoding='utf-8'
)
comparison_df.to_csv(OUTPUT_DIR / 'artpark_vs_adalat.csv', index=False)
(OUTPUT_DIR / 'artpark_vs_adalat.md').write_text(
    comparison_df.to_markdown(index=False) + '\n', encoding='utf-8'
)
paired_detail_df.to_csv(OUTPUT_DIR / 'paired_utterance_deltas.csv', index=False)
paired_summary_df.to_csv(OUTPUT_DIR / 'paired_channel_deltas.csv', index=False)
(OUTPUT_DIR / 'paired_channel_deltas.md').write_text(
    paired_summary_df.to_markdown(index=False) + '\n', encoding='utf-8'
)

print('Single-reference per-condition summary:')
display(summary_df)
print('ARTPARK vs Adalat:')
display(comparison_df)
print('Paired channel changes:')
display(paired_summary_df)


In [ ]:
# Speaker-clustered bootstrap and objective external-replication verdict.
from callwhisper.eval.paired_bootstrap import (
    paired_bootstrap_rows,
    write_outputs as write_bootstrap_outputs,
)

artpark_rows = read_jsonl(prediction_paths[0])
adalat_rows = read_jsonl(prediction_paths[1])
bootstrap_rows = paired_bootstrap_rows(
    artpark_rows,
    adalat_rows,
    replicates=BOOTSTRAP_REPLICATES,
    seed=SEED,
)
write_bootstrap_outputs(bootstrap_rows, OUTPUT_DIR)
bootstrap_df = pd.DataFrame(bootstrap_rows)

pooled_gap_row = bootstrap_df[
    (bootstrap_df['metric'] == 'channel_penalty_gap')
    & (bootstrap_df['slice'] == 'pooled_telephone')
].iloc[0]
original_gap_row = bootstrap_df[
    (bootstrap_df['metric'] == 'model_gap')
    & (bootstrap_df['slice'] == 'original')
].iloc[0]
pooled_model_gap_row = bootstrap_df[
    (bootstrap_df['metric'] == 'model_gap')
    & (bootstrap_df['slice'] == 'pooled_telephone')
].iloc[0]

if pooled_gap_row['ci_95_lower'] > 0:
    verdict = 'replicated'
    interpretation = (
        'Adalat has a statistically supported larger pooled telephone WER penalty '
        'than ARTPARK on this LAHAJA speaker-balanced slice.'
    )
elif pooled_gap_row['ci_95_upper'] < 0:
    verdict = 'reversed'
    interpretation = (
        'Adalat has a statistically supported smaller pooled telephone WER penalty '
        'than ARTPARK on this LAHAJA speaker-balanced slice.'
    )
else:
    verdict = 'inconclusive'
    interpretation = (
        'The pooled channel-penalty gap confidence interval includes zero; '
        'this external replication is inconclusive.'
    )

artpark_rtf = float(summary_df[
    (summary_df['model_label'] == model_labels[0])
    & (summary_df['slice'] == 'pooled_telephone')
]['mean_real_time_factor'].iloc[0])
adalat_rtf = float(summary_df[
    (summary_df['model_label'] == model_labels[1])
    & (summary_df['slice'] == 'pooled_telephone')
]['mean_real_time_factor'].iloc[0])

conclusion = {
    'verdict': verdict,
    'interpretation': interpretation,
    'primary_metric': 'pooled telephone channel-penalty gap',
    'channel_penalty_gap_estimate': float(pooled_gap_row['estimate']),
    'channel_penalty_gap_ci_95': [
        float(pooled_gap_row['ci_95_lower']),
        float(pooled_gap_row['ci_95_upper']),
    ],
    'original_model_gap_estimate': float(original_gap_row['estimate']),
    'original_model_gap_ci_95': [
        float(original_gap_row['ci_95_lower']),
        float(original_gap_row['ci_95_upper']),
    ],
    'pooled_model_gap_estimate': float(pooled_model_gap_row['estimate']),
    'pooled_model_gap_ci_95': [
        float(pooled_model_gap_row['ci_95_lower']),
        float(pooled_model_gap_row['ci_95_upper']),
    ],
    'artpark_pooled_rtf': artpark_rtf,
    'adalat_pooled_rtf': adalat_rtf,
    'artpark_to_adalat_rtf_ratio': artpark_rtf / adalat_rtf,
    'important_limitations': [
        'LAHAJA provides one reference per utterance; Vaani used three-reference scoring.',
        'Absolute WER across LAHAJA and Vaani is not directly comparable.',
        'Model training-set disjointness from LAHAJA is not independently proven.',
        'The paired degradation effect is causal for these transforms, not for every real phone call.',
    ],
}
(OUTPUT_DIR / 'replication_conclusion.json').write_text(
    json.dumps(conclusion, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
conclusion_md = f"""# LAHAJA External Replication Verdict

**Verdict:** {verdict}

{interpretation}

- Pooled channel-penalty gap: {pooled_gap_row['estimate']:.6f}
- 95% speaker-bootstrap CI: [{pooled_gap_row['ci_95_lower']:.6f}, {pooled_gap_row['ci_95_upper']:.6f}]
- Original Adalat-minus-ARTPARK model gap: {original_gap_row['estimate']:.6f}
- Pooled Adalat-minus-ARTPARK model gap: {pooled_model_gap_row['estimate']:.6f}
- ARTPARK/Adalat pooled RTF ratio: {artpark_rtf / adalat_rtf:.3f}

## Limits

- LAHAJA uses conventional single-reference scoring; Vaani used multi-reference scoring.
- Compare paired penalties and confidence intervals, not absolute WER across datasets.
- Training-set disjointness from LAHAJA is not independently proven.
- Results describe this frozen speaker-balanced slice and these channel transforms.
"""
(OUTPUT_DIR / 'replication_conclusion.md').write_text(
    conclusion_md, encoding='utf-8'
)

display(bootstrap_df[
    bootstrap_df['metric'].isin(['channel_penalty', 'channel_penalty_gap'])
])
print(json.dumps(conclusion, indent=2))


In [ ]:
# Final completeness audit and small report bundle. Audio archives remain separate.
required_outputs = [
    'run_config.json',
    'package_versions.json',
    'selection_summary.json',
    'validation_summary.json',
    'lahaja_speaker_132.csv',
    'lahaja_speaker_132_all_conditions.csv',
    'frozen_eval_rows.csv',
    'artpark_medium_vaani_hindi_predictions.jsonl',
    'adalat_whisper_small_hi_high_lr_predictions.jsonl',
    'summary.csv',
    'artpark_vs_adalat.csv',
    'paired_channel_deltas.csv',
    'paired_utterance_deltas.csv',
    'paired_bootstrap_v1.csv',
    'paired_bootstrap_v1.json',
    'paired_bootstrap_v1.md',
    'replication_conclusion.json',
    'replication_conclusion.md',
]
missing_outputs = [
    name for name in required_outputs if not (OUTPUT_DIR / name).exists()
]
if missing_outputs:
    raise RuntimeError(f'Run incomplete; missing outputs: {missing_outputs}')

report_archive = OUTPUT_DIR / 'lahaja_external_replication_report_v1.tar.gz'
with tarfile.open(report_archive, 'w:gz') as archive:
    for name in required_outputs:
        archive.add(OUTPUT_DIR / name, arcname=name)

print('COMPLETE')
print('Verdict:', conclusion['verdict'])
print('Report bundle:', report_archive)
print('All persistent outputs:', OUTPUT_DIR)
print('Next: return replication_conclusion.json, summary.csv, and paired_bootstrap_v1.csv.')


## Reading The Result

Use `replication_conclusion.json` first.

- **replicated**: pooled `channel_penalty_gap` 95% CI is entirely above zero. Adalat-small loses more WER under telephone transforms than ARTPARK on this external slice.
- **inconclusive**: CI crosses zero. Current external sample does not establish different channel sensitivity.
- **reversed**: CI is entirely below zero. Adalat-small is more channel-robust on this slice.

Negative or positive raw WER changes for individual conditions can happen by chance. Primary claim comes from paired speaker bootstrap, not one condition's point estimate.

Do not compare LAHAJA's absolute single-reference WER directly with Vaani's multi-reference WER. If result replicates, next experiment is channel-balanced adaptation of compact Whisper-small with clean replay, followed by one frozen re-evaluation on both benchmarks.
